# 🚀 Aprendizaje Multi-Etapa con el Arnés (Preentrenamiento + Fine-Tuning + Alineación)

> **Objetivo:** Comprender de forma 100% práctica y divulgativa cómo se entrenan los modelos de IA modernos (como GPT o LLaMA) combinando diferentes paradigmas de aprendizaje en etapas consecutivas utilizando el **arnés de experimentos (`harness.py`)**.

---

### El Pipeline de 3 Etapas que construiremos:

```none
┌────────────────────────────────────────────────────────────────────────┐
│ 1. PREENTRENAMIENTO (Autosupervisado / Next-Token Prediction)          │
│    • Entrada: Frases crudas sin etiquetar.                             │
│    • Objetivo: Aprender gramática y relaciones semánticas.             │
│    • Resultado: Guardado inmutable en `runs/fase1_pretrain/weights.pt` │
├────────────────────────────────────────────────────────────────────────┤
│ 2. FINE-TUNING (Supervisado / Clasificación de Sentimiento)            │
│    • Entrada: Muy pocos datos etiquetados (ej. 4 críticas de producto).│
│    • Conexión: Carga los pesos del encoder aprendidos en la Fase 1.    │
│    • Resultado: Clasificador preciso sin necesidad de miles de datos.  │
├────────────────────────────────────────────────────────────────────────┤
│ 3. ALINEACIÓN POR RECOMPENSA (Aprendizaje por Refuerzo / Policy)       │
│    • Entrada: Generación libre del modelo evaluada por una recompensa. │
│    • Objetivo: Guiar las probabilidades hacia respuestas amables/útiles│
└────────────────────────────────────────────────────────────────────────┘
```

## 0. Arranque del entorno y registro de optimizadores

In [ ]:
# ── ARRANQUE ──
import os, sys
from pathlib import Path

while not (Path.cwd() / "lab").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from lab import harness as H

# Registrar optimizadores estándar en el arnés si no están aún
@H.optimizers.register("adam")
def build_adam(params, lr=1e-3, **kwargs):
    return torch.optim.Adam(params, lr=lr, **kwargs)

@H.optimizers.register("sgd")
def build_sgd(params, lr=1e-2, momentum=0.9, **kwargs):
    return torch.optim.SGD(params, lr=lr, momentum=momentum, **kwargs)

print("✅ Arnés y optimizadores listos en:", Path.cwd())

## 1. Vocabulario y Datasets Didácticos

Definimos un vocabulario reducido de palabras sobre reseñas de productos y servicios.

In [ ]:
# Vocabulario didáctico
VOCAB = [
    "<pad>", "<unk>", "el", "servicio", "producto", "es", "excelente", 
    "bueno", "malo", "pesimo", "muy", "y", "calidad", "una", "estafa", "maravilla"
]
vocab_to_id = {w: i for i, w in enumerate(VOCAB)}
id_to_vocab = {i: w for i, w in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)
SEQ_LEN = 8

print(f"Vocabulario de {VOCAB_SIZE} palabras: {VOCAB}")

# ── A. DATASET AUTOSUPERVISADO (Frases crudas sin etiquetas) ──
@H.datasets.register("unlabelled_reviews")
def build_unlabelled_reviews(batch_size=4, **kwargs):
    raw_corpus = [
        "el servicio es excelente y muy bueno",
        "el producto es de muy buena calidad",
        "el servicio es pesimo y una estafa",
        "el producto es malo y muy pesimo",
        "una maravilla el producto es excelente",
        "muy mala calidad el servicio es malo",
        "el producto es una maravilla excelente",
        "el servicio es pesimo y malo"
    ]
    sequences = []
    for line in raw_corpus:
        tokens = [vocab_to_id.get(w, 1) for w in line.split()][:SEQ_LEN]
        tokens += [0] * (SEQ_LEN - len(tokens))
        sequences.append(tokens)
    
    tensor_data = torch.tensor(sequences, dtype=torch.long)
    # Desfase temporal: X es el texto hasta penúltimo token, Y es el texto desde el 2º token
    inputs, targets = tensor_data[:, :-1], tensor_data[:, 1:]
    
    dataset = TensorDataset(inputs, targets)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# ── B. DATASET SUPERVISADO (Solo 4 frases con etiqueta humana 0 o 1) ──
@H.datasets.register("few_shot_sentiment")
def build_few_shot_sentiment(batch_size=2, **kwargs):
    labelled_data = [
        ("el servicio es excelente y una maravilla", 1),  # Positivo
        ("el producto es excelente y muy bueno", 1),      # Positivo
        ("el servicio es pesimo y una estafa", 0),        # Negativo
        ("el producto es malo y pesimo", 0),             # Negativo
    ]
    inputs_list, targets_list = [], []
    for line, label in labelled_data:
        tokens = [vocab_to_id.get(w, 1) for w in line.split()][:SEQ_LEN]
        tokens += [0] * (SEQ_LEN - len(tokens))
        inputs_list.append(tokens)
        targets_list.append(label)
        
    x = torch.tensor(inputs_list, dtype=torch.long)
    y = torch.tensor(targets_list, dtype=torch.long)
    
    dataset = TensorDataset(x, y)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

print("✅ Datasets 'unlabelled_reviews' y 'few_shot_sentiment' registrados")

## 2. Modelos Modulares con Transferencia de Pesos

Definimos una arquitectura con un **Backbone compartido** (Embedding + Encoder) que se utilizará tanto en la Fase 1 como en la Fase 2.

In [ ]:
# ── MODELO FASE 1: Modelo de Lenguaje (Autosupervisado) ──
@H.models.register("lm_pretrainer")
def build_lm_model(embed_dim=16, hidden_dim=32, **kwargs):
    class LanguageModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.embedding = nn.Embedding(VOCAB_SIZE, embed_dim, padding_idx=0)
            self.encoder = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim),
                nn.ReLU()
            )
            self.head = nn.Linear(embed_dim, VOCAB_SIZE) # Predice la siguiente palabra
            
        def forward(self, x):
            embeds = self.embedding(x)      # [B, T, embed_dim]
            h = self.encoder(embeds)        # [B, T, embed_dim]
            logits = self.head(h)           # [B, T, VOCAB_SIZE]
            return logits.transpose(1, 2)   # [B, VOCAB_SIZE, T] para CrossEntropyLoss
            
    return LanguageModel()

# ── MODELO FASE 2: Clasificador con Carga de Pesos Previos ──
@H.models.register("sentiment_classifier")
def build_sentiment_classifier(embed_dim=16, hidden_dim=32, pretrained_weights=None, **kwargs):
    class SentimentClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.embedding = nn.Embedding(VOCAB_SIZE, embed_dim, padding_idx=0)
            self.encoder = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim),
                nn.ReLU()
            )
            self.classifier = nn.Linear(embed_dim, 2) # 2 clases: 0 (Negativo), 1 (Positivo)
            
        def forward(self, x):
            embeds = self.embedding(x)
            h = self.encoder(embeds)
            pooled = h.mean(dim=1)             # Resumen de la frase (Global Average Pooling)
            return self.classifier(pooled)     # [B, 2] Logits
            
    model = SentimentClassifier()
    
    # 🔗 Transferencia de conocimiento: cargar pesos del backbone de la Fase 1
    if pretrained_weights is not None:
        saved_state = torch.load(pretrained_weights)
        model_state = model.state_dict()
        
        # Transferir embedding y encoder (omitir la cabeza generativa 'head')
        transferred = {k: v for k, v in saved_state.items() if k in model_state and "head" not in k}
        model_state.update(transferred)
        model.load_state_dict(model_state)
        print(f"📥 [Transferencia] Cargados con éxito {len(transferred)} bloques de pesos desde: {pretrained_weights}")
        
    return model

print("✅ Modelos 'lm_pretrainer' y 'sentiment_classifier' registrados")

## 3. FASE 1: Preentrenamiento Autosupervisado con el Arnés

Lanzamos el entrenamiento base sobre el texto sin etiquetar. La red aprende a predecir la siguiente palabra.

In [ ]:
config_fase1 = {
    "name": "etapa1_pretrain",
    "dataset": "unlabelled_reviews",
    "model": "lm_pretrainer",
    "loss": "cross_entropy",
    "optimizer": "adam",
    "optimizer_args": {"lr": 0.01},
    "epochs": 25,
    "seed": 42
}

print("🚀 Ejecutando FASE 1: Preentrenamiento Autosupervisado...")
res_fase1 = H.run_experiment(config_fase1)
print(f"\n💾 Checkpoint inmutable guardado en: runs/{res_fase1.run_id}/weights.pt")

## 4. FASE 2: Fine-Tuning Supervisado con el Arnés

Ajustamos el modelo en la tarea de clasificación de sentimiento usando **solo 4 frases etiquetadas**.
Le pasamos la ruta del checkpoint guardado por la Fase 1 en `model_args`.

In [ ]:
config_fase2 = {
    "name": "etapa2_finetune",
    "dataset": "few_shot_sentiment",
    "model": "sentiment_classifier",
    "model_args": {
        # 🔗 Enlace explícito de procedencia hacia el checkpoint de la Fase 1
        "pretrained_weights": f"runs/{res_fase1.run_id}/weights.pt"
    },
    "loss": "cross_entropy",
    "optimizer": "adam",
    "optimizer_args": {"lr": 0.005}, # Tasa de aprendizaje más baja para no destruir el conocimiento
    "epochs": 15,
    "seed": 42
}

print("🚀 Ejecutando FASE 2: Fine-Tuning Supervisado...")
res_fase2 = H.run_experiment(config_fase2)
print(f"\n💾 Checkpoint clasificador guardado en: runs/{res_fase2.run_id}/weights.pt")

## 5. Experimento de Control: ¿Qué pasa si NO hubiéramos preentrenado?

Entrenamos el clasificador **desde cero (Scratch)** con los mismos 4 datos para contrastar la ventaja del preentrenamiento.

In [ ]:
config_scratch = {
    "name": "etapa2_scratch_sin_pretrain",
    "dataset": "few_shot_sentiment",
    "model": "sentiment_classifier",
    "model_args": {
        "pretrained_weights": None  # Pesos aleatorios desde cero
    },
    "loss": "cross_entropy",
    "optimizer": "adam",
    "optimizer_args": {"lr": 0.005},
    "epochs": 15,
    "seed": 42
}

print("🧪 Ejecutando Experimento de Control (Desde cero sin preentrenamiento)...")
res_scratch = H.run_experiment(config_scratch)

## 6. Comparación y Gráficas con el Arnés

Utilizamos las utilidades `compare_runs` y `plot_runs` del arnés para verificar la diferencia.

In [ ]:
# Tabla comparativa
runs_to_compare = [res_fase1.run_id, res_fase2.run_id, res_scratch.run_id]
df = H.compare_runs(runs_to_compare)
display(df[["run_id", "name", "final", "best", "seconds"]])

# Gráfica de curvas de aprendizaje
fig, ax = plt.subplots(figsize=(8, 4.5))
H.plot_runs([res_fase2.run_id, res_scratch.run_id], metrics=["val_loss"], log_scale=False, ax=ax)
ax.set_title("Fine-Tuning con Preentrenamiento vs. Entrenamiento Desde Cero")
ax.set_ylabel("Pérdida en Validación")
plt.tight_layout()
plt.show()

## 7. FASE 3: Demostración de Alineación por Recompensa / Refuerzo (RL)

En esta fase ilustrativa, mostramos cómo una función de recompensa escalar puede guiar al modelo para favorecer respuestas amables frente a respuestas tóxicas.

In [ ]:
# Modelo de política simple para demostración de RL
policy_model = res_fase2.model
policy_model.eval()

# Frase de prueba
test_phrase = "el servicio es excelente y muy bueno"
tokens = torch.tensor([[vocab_to_id.get(w, 1) for w in test_phrase.split()][:SEQ_LEN] + [0]*(SEQ_LEN - len(test_phrase.split()))])

with torch.no_grad():
    logits = policy_model(tokens)
    probs = F.softmax(logits, dim=-1)[0]

print(f"• Frase evaluada: '{test_phrase}'")
print(f"  → Probabilidad Negativo: {probs[0]*100:.1f}%")
print(f"  → Probabilidad Positivo: {probs[1]*100:.1f}%")

# Simulación de Recompensa (RLHF Policy Reward):
# Si clasifica correctamente el elogio recibe Recompensa +1.0, si falla recibe -1.0
recompensa = 1.0 if probs.argmax().item() == 1 else -1.0
print(f"\n🏆 Recompensa obtenida del evaluador (Reward R): {recompensa:+.1f}")
print(f"   Fórmula de pérdida de política: Loss = -log(prob) * Reward = {-torch.log(probs[1]).item() * recompensa:.4f}")

---

## 💡 Conclusiones del Aprendizaje Multi-Etapa con el Arnés

1. **Desacoplamiento total:** Cada etapa se ejecutó de forma independiente con su propio `config` y `run_id`, dejando una trazabilidad limpia en la carpeta `runs/`.
2. **Reutilización de cómputo:** El preentrenamiento se ejecutó una sola vez y se reutilizó sin repetir cómputo.
3. **Eficiencia de datos:** Con solo 4 ejemplos etiquetados, el modelo con preentrenamiento aprendió significativamente más rápido que el entrenado desde cero.